# 03 · Reliable producers: acknowledgements, errors and batching
**Level:** beginner → intermediate · **Time:** 40 minutes · **Prerequisite:** 02.
Distinguish local enqueue, successful delivery and business success. Handle malformed input,
observe an actual broker delivery failure, and attach trace metadata.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
require_broker()
topic = new_topic('producer-reliability', partitions=1)


## 1. Name your guarantees
Our producer explicitly enables idempotence and `acks=all`. Idempotence protects against
duplicates caused by protocol retries within the producer session. Sending the same business
event twice is still two sends. `acks=all` acknowledges the in-sync replica set; with one
replica it is not protection against losing that broker's disk.


In [ ]:
import json
delivery_log = []
p = producer(**{"linger.ms": 10, "compression.type": "gzip"})
def report(error, message):
    delivery_log.append({"error": str(error) if error else None,
                         "partition": message.partition(), "offset": message.offset()})
for sequence in range(10):
    event = {"event_id": str(uuid4()), "order_id": "order-batched", "sequence": sequence}
    p.produce(topic, key=b"order-batched", value=json.dumps(event).encode(),
              headers=[("trace-id", RUN.encode()), ("content-type", b"application/json")],
              on_delivery=report)
    p.poll(0)
print("Queued ten calls; final delivery status follows in the next cell.")


## 2. Drain and inspect both signals
Batching amortizes network overhead. Flushing after every record would defeat much of
the batching. A full local queue raises `BufferError`; poll with a deadline and retry the
enqueue, while leaving delivery retries to the client.


In [ ]:
pending = p.flush(15)
failures = [entry for entry in delivery_log if entry["error"]]
assert pending == 0 and not failures, (pending, failures)
assert len(delivery_log) == 10
delivery_log


## 3. Read trace headers
Headers are metadata; Kafka does not validate their meaning. Avoid putting passwords
or sensitive personal data into trace metadata.


In [ ]:
messages = collect(topic, count=10)
print(messages[0].headers())
assert dict(messages[0].headers())["trace-id"] == RUN.encode()
assert [decode(m)["sequence"] for m in messages] == list(range(10))


## 4. Fail before enqueueing bad JSON
Validate and serialize before calling the producer. This error is local; it has no
delivery report because the producer never received bytes.


In [ ]:
from decimal import Decimal
try:
    json.dumps({"total": Decimal("19.99")})
except TypeError as error:
    print("Expected serialization failure:", error)
fixed = {"total": str(Decimal("19.99"))}
assert json.loads(json.dumps(fixed)) == {"total": "19.99"}


## 5. Trigger a real delivery failure without stopping the broker
Create an isolated topic with `min.insync.replicas=2`, although it has only one replica.
An `acks=all` producer cannot satisfy this requirement. The failure is deliberately bounded.
This demonstrates the configuration constraint; notebook 15 tests actual replica failure.


In [ ]:
strict_topic = new_topic("insufficient-isr", partitions=1,
                         config={"min.insync.replicas": "2"})
bad_reports = []
strict_producer = producer(**{"delivery.timeout.ms": 4000, "request.timeout.ms": 1000})
def failed_delivery(error, message):
    bad_reports.append(str(error) if error else None)
strict_producer.produce(strict_topic, value=b'{"example":"cannot-ack"}', on_delivery=failed_delivery)
pending = strict_producer.flush(8)
assert pending == 0 and len(bad_reports) == 1 and bad_reports[0] is not None, bad_reports
print("Expected delivery error:", bad_reports[0])


## 6. Business duplicates still need business handling
A network retry and a second application `produce()` call are different things.


In [ ]:
duplicate_topic = new_topic("application-duplicates", partitions=1)
same_event = {"event_id": str(uuid4()), "order_id": "order-duplicate", "total": "12.00"}
publish(duplicate_topic, [same_event, same_event], key_field="order_id")
duplicates = collect(duplicate_topic, count=2)
assert decode(duplicates[0])["event_id"] == decode(duplicates[1])["event_id"]
assert duplicates[0].offset() != duplicates[1].offset()
print("Two sends created two records. Django lab 12 will deduplicate their effects.")


## Exercise
Predict the result if you only check `flush() == 0` in step 5. Add a callback that counts
successes and failures separately. What should an HTTP endpoint report if its database
commit succeeded but its Kafka delivery failed?
<details><summary>Solution hints</summary>
The queue can drain with failures. Inspect both pending count and delivery errors.
A DB commit cannot be undone by a later Kafka failure; use an outbox and return the
stored order's status. A blind HTTP retry can create another order without request idempotency.
</details>
Next: **04 — offsets, commits and replay**.
